# Joining Customer tables 
# from two sources on SURROGATE KEY

# Create the CRM DataFrame

In [1]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.getOrCreate()

crm = spark.createDataFrame([
    (1, "Robert Smith", "Bob@gmail.com"),
    (2, "Mell Brown", "Mell@gmail.com"),
    (3, "Otto Jones", "Otto@gmail.com")
], ["CRM_ID", "CRM_Name", "Email"])

crm.show()

StatementMeta(, e72d30f8-227c-4b1e-8b21-581e080df71c, 3, Finished, Available, Finished, False)

+------+------------+--------------+
|CRM_ID|    CRM_Name|         Email|
+------+------------+--------------+
|     1|Robert Smith| Bob@gmail.com|
|     2|  Mell Brown|Mell@gmail.com|
|     3|  Otto Jones|Otto@gmail.com|
+------+------------+--------------+



# Create the Online Shop DataFrame

In [2]:
shop = spark.createDataFrame([
    ("A125F", "Rober Smith", "Bob@gmail.com"),
    ("B772X", "Mell Brown", "Mell@gmail.com"),
    ("C998L", "Otto Jones", "Otto@gmail.com")
], ["SHOP_ID", "Name", "Email"])

shop.show()

StatementMeta(, e72d30f8-227c-4b1e-8b21-581e080df71c, 4, Finished, Available, Finished, False)

+-------+-----------+--------------+
|SHOP_ID|       Name|         Email|
+-------+-----------+--------------+
|  A125F|Rober Smith| Bob@gmail.com|
|  B772X| Mell Brown|Mell@gmail.com|
|  C998L| Otto Jones|Otto@gmail.com|
+-------+-----------+--------------+



# CRM and Online Shop 
# - Customer tables

In [3]:
crm.show()
shop.show()

StatementMeta(, e72d30f8-227c-4b1e-8b21-581e080df71c, 6, Finished, Available, Finished, False)

+------+------------+--------------+
|CRM_ID|    CRM_Name|         Email|
+------+------------+--------------+
|     1|Robert Smith| Bob@gmail.com|
|     2|  Mell Brown|Mell@gmail.com|
|     3|  Otto Jones|Otto@gmail.com|
+------+------------+--------------+

+-------+-----------+--------------+
|SHOP_ID|       Name|         Email|
+-------+-----------+--------------+
|  A125F|Rober Smith| Bob@gmail.com|
|  B772X| Mell Brown|Mell@gmail.com|
|  C998L| Otto Jones|Otto@gmail.com|
+-------+-----------+--------------+



# Wrong way - joining on the IDs
# CRM_ID ≠ SHOP_ID

In [4]:
crm.join(shop, crm.CRM_ID == shop.SHOP_ID).show()

StatementMeta(, e72d30f8-227c-4b1e-8b21-581e080df71c, 8, Finished, Available, Finished, False)

+------+--------+-----+-------+----+-----+
|CRM_ID|CRM_Name|Email|SHOP_ID|Name|Email|
+------+--------+-----+-------+----+-----+
+------+--------+-----+-------+----+-----+



# Join on the Business Key

In [5]:
merged = crm.join(shop, on="Email", how="inner")

merged.show()

StatementMeta(, e72d30f8-227c-4b1e-8b21-581e080df71c, 9, Finished, Available, Finished, False)

+--------------+------+------------+-------+-----------+
|         Email|CRM_ID|    CRM_Name|SHOP_ID|       Name|
+--------------+------+------------+-------+-----------+
| Bob@gmail.com|     1|Robert Smith|  A125F|Rober Smith|
|Mell@gmail.com|     2|  Mell Brown|  B772X| Mell Brown|
|Otto@gmail.com|     3|  Otto Jones|  C998L| Otto Jones|
+--------------+------+------------+-------+-----------+



# Create a New Surrogate Key

In [6]:
from pyspark.sql.functions import monotonically_increasing_id

dw_customers = (
    merged
    .select("CRM_ID", "SHOP_ID", "Name", "Email")
    .withColumn("DWCustomerKey", monotonically_increasing_id() + 1)
)

dw_customers.show()

StatementMeta(, e72d30f8-227c-4b1e-8b21-581e080df71c, 13, Finished, Available, Finished, False)

+------+-------+-----------+--------------+-------------+
|CRM_ID|SHOP_ID|       Name|         Email|DWCustomerKey|
+------+-------+-----------+--------------+-------------+
|     1|  A125F|Rober Smith| Bob@gmail.com|            1|
|     2|  B772X| Mell Brown|Mell@gmail.com|            2|
|     3|  C998L| Otto Jones|Otto@gmail.com|            3|
+------+-------+-----------+--------------+-------------+

